In [ ]:
import pickle, math, random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from model_arch import *

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_ID, EOS_ID = 0, 3

In [ ]:
train = pickle.load(open("tok_train.pkl", "rb"))
dev = pickle.load(open("tok_dev.pkl", "rb"))

model = DecoderLM(LMConfig(vocab_size=50260))
model.load_state_dict(torch.load("dec.pt", map_location="cpu"))
model.to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
def make_chunks(doc, max_chunks=None):
    n = math.ceil(len(doc) / 1023)
    idxs = list(range(n))
    if max_chunks is not None and n > max_chunks:
        idxs = sorted(random.sample(idxs, max_chunks))
    
    chunks = []
    for i in idxs:
        piece = doc[i * 1023:(i + 1) * 1023].astype(np.int64)
        chunks.append(np.append(piece, EOS_ID))
    return chunks

def chunk_vectors(model, chunks):
    lengths = torch.tensor([len(c) for c in chunks], device=DEVICE)
    T = int(lengths.max())
    ids = torch.full((len(chunks), T), PAD_ID, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c)
    ids = ids.to(DEVICE)

    with torch.autocast("cuda", dtype=torch.float16):
        x = model.tok_emb(ids) + model.pos_emb[:T]
        for block in model.blocks:
            x = block(x)
        x = model.ln_f(x)

    last = x[torch.arange(len(chunks), device=DEVICE), lengths - 1]
    return F.normalize(last.float(), dim=-1)

def doc_vectors(model, docs, max_chunks=None, chunk_batch=64):
    all_chunks, owner = [], []
    for d_i, doc in enumerate(docs):
        cs = make_chunks(doc, max_chunks)
        all_chunks += cs
        owner += [d_i] * len(cs)
    
    owner = torch.tensor(owner, device=DEVICE)
    sums = torch.zeros(len(docs), model.cfg.d_model, device=DEVICE)
    
    for s in range(0, len(all_chunks), chunk_batch):
        v = chunk_vectors(model, all_chunks[s:s + chunk_batch])
        sums = sums.index_add(0, owner[s:s + chunk_batch], v)
        
    return F.normalize(sums, dim=-1)

In [ ]:
@torch.no_grad()
def evaluate(model, data):
    model.eval()
    cand_ids = list(data["cands"].keys())
    cand_vecs = doc_vectors(model, [data["cands"][c] for c in cand_ids])
    
    q_ids = list(data["labels"].keys())
    q_vecs = doc_vectors(model, [data["queries"][q] for q in q_ids])

    sims = q_vecs @ cand_vecs.T
    col = {c: i for i, c in enumerate(cand_ids)}
    for r, q in enumerate(q_ids):
        if q in col:
            sims[r, col[q]] = float("-inf")
            
    top7 = sims.topk(7, dim=1).indices.cpu().numpy()

    tp = n_gold = 0
    for r, q in enumerate(q_ids):
        predicted = {cand_ids[j] for j in top7[r]}
        gold = set(data["labels"][q])
        tp += len(predicted & gold)
        n_gold += len(gold)
        
    P = tp / (len(q_ids) * 7)
    R = tp / n_gold
    F1 = 2 * P * R / (P + R) if P + R > 0 else 0.0
    return P, R, F1

In [ ]:
pairs = [(q, c) for q, rels in train["labels"].items() for c in rels if c in train["cands"]]
B = 8
steps_per_epoch = len(pairs) // B

best = 0.0

for epoch in range(1, 4):
    model.train()
    random.shuffle(pairs)
    running = 0.0

    for step in range(steps_per_epoch):
        batch = pairs[step * B:(step + 1) * B]
        q_docs = [train["queries"][q] for q, _ in batch]
        c_docs = [train["cands"][c] for _, c in batch]

        vecs = doc_vectors(model, q_docs + c_docs, max_chunks=4, chunk_batch=10_000)
        qv, cv = vecs[:B], vecs[B:]
        sim = qv @ cv.T / 0.05
        
        mask = torch.zeros(B, B, dtype=torch.bool)
        for i, (qi, _) in enumerate(batch):
            rel = set(train["labels"][qi])
            for j, (_, cj) in enumerate(batch):
                if i != j and cj in rel:
                    mask[i, j] = True
        sim = sim.masked_fill(mask.to(DEVICE), float("-inf"))
        loss = F.cross_entropy(sim, torch.arange(B, device=DEVICE))

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        running += loss.item()

    P, R, F1 = evaluate(model, dev)
    print("Dev P@7", P, "R@7", R, "F1@7", F1)

    torch.save(model.state_dict(), f"epoch{epoch}.pt")